# 📋 [OPS08_M04] 가상 드론 경보 발령 시 역할별 초동 조치 및 보고 체계 1차 모의훈련(Dry-run)
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (정답 & 교관 가이드북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m04_workbook.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M04
- **미션 주제:** 가상 드론 경보 발령 시 역할별 초동 조치 및 보고 체계 1차 모의훈련(Dry-run)
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 04. 가상 드론 경보 발령 시 역할별 초동 조치 및 보고 체계 1차 모의훈련(Dry-run)

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"1차 Dry-run 10개 사건의 초 단위 타임스탬프 기록을 바탕으로 평균 지연 시간과 SOP 준수율(%)을 계산하고, 가장 심각한 병목 구간 2건을 도출해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 타임스탬프 계산이 원시 데이터와 일치하는지 교차 검산하고 구조적 통신 병목 원인을 검증합니다.

---

### 📑 실습자료 및 과업 지침

가상 드론 경보 발령 시 역할별 초동 조치 및 보고 체계 1차 모의훈련(Dry-run)**

가. 제공 자료 - 1차 모의훈련 진행 시간표(권장)

| **시간** | **활동** | **담당** |
| --- | --- | --- |
| **0~10분** | 역할 카드 배부, 안전 브리핑, 초시계 동기화 | 교관 |
| **10~15분** | SOP·요약 카드 개인 숙지 | 전원 |
| **15~45분** | 주입 카드 1~10번 순서 진행(1차 Dry-run) | 교관·전원 |
| **45~60분** | 기록표 정리, 준수율·지연시간 계산 | 기록자·평가관 |
| **60~75분** | 병목·혼선 구간 토의 | 전원 |

나. 작성 서식 1 - 시각 기록표 (기록자, 초 단위)

| **카드** | **최초 포착** | **1차 보고** | **지휘자 수신** | **전파** | **대피 완료** | **보고 지연(초)
=수신-포착** | **조치 시간(초)
=대피-포착** |
| --- | --- | --- | --- | --- | --- | --- | --- |
| **예시** | 05:43:05 | 05:43:32 | 05:43:40 | 05:43:55 | 05:44:30 | 35 | 85 |
| **1** |  |  |  |  |  |  |  |
| **2** |  |  |  |  |  |  |  |
| **3** |  |  |  |  |  |  |  |
| **4** |  |  |  |  |  |  |  |
| **5** |  |  |  |  |  |  |  |
| **6** |  |  |  |  |  |  |  |
| **7** |  |  |  |  |  |  |  |
| **8** |  |  |  |  |  |  |  |
| **9** |  |  |  |  |  |  |  |
| **10** |  |  |  |  |  |  |  |

다. 작성 서식 2 - SOP 단계 수행 체크(평가관)

| **카드** | **① 포착** | **② 식별** | **③ 보고** | **④ 전파** | **⑤ 결심** | **⑥ 대피·대응** | **⑦ 해제** | **수행/요구** |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **1** |  |  |  |  |  |  |  |  |
| **2** |  |  |  |  |  |  |  |  |
| **3** |  |  |  |  |  |  |  |  |
| **4** |  |  |  |  |  |  |  |  |
| **5** |  |  |  |  |  |  |  |  |
| **6** |  |  |  |  |  |  |  |  |
| **7** |  |  |  |  |  |  |  |  |
| **8** |  |  |  |  |  |  |  |  |
| **9** |  |  |  |  |  |  |  |  |
| **10** |  |  |  |  |  |  |  |  |

| **SOP 준수율** | 수행한 단계 합계 ( ) ÷ 요구된 단계 합계 ( ) × 100 = ( )% → 목표 80% 이상 ( ) |
| --- | --- |

| **평균 보고 지연** | 보고 지연 합계 ( )초 ÷ 카드 수 ( ) = ( )초 → 목표 180초(3분) 이내 ( ) |
| --- | --- |

라. 작성 서식 3 - 참여자별 수행 기록지(각자 작성)

| **역할** | **잘한 점** | **막힌 점(카드 번호)** | **SOP에 없어서 헷갈린 것** |
| --- | --- | --- | --- |
| **관측자** |  |  |  |
| **지휘자** |  |  |  |
| **경계조** |  |  |  |
| **통신·기록** |  |  |  |

마. 작성 서식 4 - 병목·혼선 구간 기록

| **번호** | **발생 단계·카드** | **현상** | **추정 원인** | **M8로 넘길 질문** |
| --- | --- | --- | --- | --- |
| **1** |  |  |  |  |
| **2** |  |  |  |  |
| **3** |  |  |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 모든 카드의 시각이 초 단위로 기록되었다 | ( ) |
| SOP 준수율을 계산식대로 계산했다 | ( ) |
| 평균 보고 지연을 계산했다 | ( ) |
| 참여자 전원의 수행 기록지를 수합했다 | ( ) |
| 병목·혼선 구간을 2건 이상 적었다 | ( ) |
| 훈련 중 안전 위반이 없었다 | ( ) |

**



In [ ]:
# [미션 4 실습 코드] 1차 모의훈련 타임라인 및 SOP 준수율 계산
data = [
    {'event_id': 'MSEL-01', 'detect_sec': 5, 'report_sec': 24, 'relay_sec': 48, 'action_sec': 78, 'lead_time_sec': 78, 'sop_pass': True},
    {'event_id': 'MSEL-02', 'detect_sec': 12, 'report_sec': 45, 'relay_sec': 95, 'action_sec': 140, 'lead_time_sec': 140, 'sop_pass': True},
    {'event_id': 'MSEL-03', 'detect_sec': 4, 'report_sec': 15, 'relay_sec': 32, 'action_sec': 55, 'lead_time_sec': 55, 'sop_pass': True},
    {'event_id': 'MSEL-04', 'detect_sec': 8, 'report_sec': 30, 'relay_sec': 70, 'action_sec': 110, 'lead_time_sec': 110, 'sop_pass': True},
    {'event_id': 'MSEL-05', 'detect_sec': 6, 'report_sec': 65, 'relay_sec': 120, 'action_sec': 185, 'lead_time_sec': 185, 'sop_pass': False},
    {'event_id': 'MSEL-06', 'detect_sec': 20, 'report_sec': 80, 'relay_sec': 140, 'action_sec': 210, 'lead_time_sec': 210, 'sop_pass': False},
    {'event_id': 'MSEL-07', 'detect_sec': 3, 'report_sec': 12, 'relay_sec': 28, 'action_sec': 49, 'lead_time_sec': 49, 'sop_pass': True},
    {'event_id': 'MSEL-08', 'detect_sec': 7, 'report_sec': 95, 'relay_sec': 160, 'action_sec': 240, 'lead_time_sec': 240, 'sop_pass': False},
    {'event_id': 'MSEL-09', 'detect_sec': 9, 'report_sec': 25, 'relay_sec': 52, 'action_sec': 88, 'lead_time_sec': 88, 'sop_pass': True},
    {'event_id': 'MSEL-10', 'detect_sec': 5, 'report_sec': 28, 'relay_sec': 65, 'action_sec': 105, 'lead_time_sec': 105, 'sop_pass': True}
]
log_df = pd.DataFrame(data)
analyze_mission4_dryrun(log_df)
